# Track B: Laptop Price Regression — Startech.com.bd

**Target**: `Price_BDT` (BDT)  \n**Source**: Scraped from [startech.com.bd](https://www.startech.com.bd/laptop-notebook/laptop)  
**Pipeline**: Data Cleaning → EDA → Feature Engineering → Modeling → Evaluation  
**Global seed**: `random_state=42`

## 1. Data Cleaning

### 1.1 Load raw scraped data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import re, warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
np.random.seed(42)

RANDOM_STATE = 42
RAW_PATH  = "../data/track_b/raw/track_b_listings.csv"
CLEANED_PATH = "../data/track_b/cleaned/laptop_listings_cleaned.csv"

df_raw = pd.read_csv(RAW_PATH, encoding='utf-8-sig')
print(f"Shape: {df_raw.shape}")
df_raw.head(3)

### 1.2 Data Dictionary

In [ ]:
dtypes = df_raw.dtypes.astype(str)
missing_ct = df_raw.isnull().sum()
missing_pct = (df_raw.isnull().mean() * 100).round(1)
nunique = df_raw.nunique()
dict_df = pd.DataFrame({
    "Dtype": dtypes, "Missing": missing_ct,
    "Missing%": missing_pct, "Unique": nunique
})
dict_df

### 1.3 Strip Taka symbol and clean price

In [ ]:
df = df_raw.copy()
# Price_BDT is already numeric from the scraper (float64)
# Check for NaN prices (out-of-stock items)
print(f"Price_BDT missing: {df['Price_BDT'].isna().sum()} out of {len(df)}")
print("Rows with missing price:")
missing_prices = df[df['Price_BDT'].isna()]
if len(missing_prices) > 0:
    print(missing_prices[["Model", "Price_BDT", "Availability"]].head(10).to_string())
print("Decision: Rows with non-numeric Price (Out of Stock) will be dropped — cannot use as target.")

### 1.4 Deduplicate listings

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=["Product_URL"], keep="first")
print(f"Dedup: {before} \u2192 {len(df)} rows")

### 1.5 Parse RAM \u2192 GB

In [ ]:
def parse_ram_gb(text):
    if pd.isna(text): return np.nan
    text = str(text)
    match = re.search(r'(\d+)\s*GB', text, re.IGNORECASE)
    if match: return int(match.group(1))
    match = re.search(r'(\d+)\s*MB', text, re.IGNORECASE)
    if match: return int(match.group(1)) // 1024
    return np.nan

df["RAM_GB"] = df["RAM"].apply(parse_ram_gb)
print(f"RAM missing after parse: {df['RAM_GB'].isna().sum()} / {len(df)}")

### 1.6 Parse Storage \u2192 SSD_GB, HDD_GB

In [ ]:
def parse_storage(text):
    if pd.isna(text): return (0, 0)
    text = str(text)
    if "storage:" in text.lower():
        text = text.lower().split("storage:")[1].strip()
    ssd, hdd = 0.0, 0.0
    def _to_gb(part):
        nums = re.findall(r'[\d.]+', part)
        if not nums: return 0.0
        val = float(nums[-1])
        if "tb" in part.lower(): val *= 1024
        return val
    lower = text.lower()
    if "ssd" in lower:
        ssd = _to_gb(text[:text.lower().index("ssd")])
    if "hdd" in lower:
        hdd = _to_gb(text[:text.lower().index("hdd")])
    if ssd == 0 and hdd == 0:
        total = _to_gb(text)
        if total > 0: ssd = total
    return (int(round(ssd)), int(round(hdd)))

storage_parsed = df["Storage"].apply(parse_storage)
df["SSD_GB"] = storage_parsed.apply(lambda x: x[0])
df["HDD_GB"] = storage_parsed.apply(lambda x: x[1])
df["Total_Storage_GB"] = df["SSD_GB"] + df["HDD_GB"]
print("Storage parsing samples:")
print(df[["Storage", "SSD_GB", "HDD_GB"]].head(10))

### 1.7 Parse Processor \u2192 brand + GHz

In [ ]:
def parse_processor(text):
    if pd.isna(text): return ("Other", 0.0)
    lower = str(text).lower()
    if "intel" in lower: brand = "Intel"
    elif "amd" in lower or "ryzen" in lower: brand = "AMD"
    elif "apple" in lower or "m1" in lower or "m2" in lower: brand = "Apple"
    elif "qualcomm" in lower or "snapdragon" in lower: brand = "Qualcomm"
    elif "mediatek" in lower: brand = "MediaTek"
    else: brand = "Other"
    ghz_match = re.search(r'([\d.]+)\s*ghz', lower)
    ghz = float(ghz_match.group(1)) if ghz_match else 0.0
    return (brand, ghz)

proc_parsed = df["Processor"].apply(parse_processor)
df["CPU_Brand"] = proc_parsed.apply(lambda x: x[0])
df["CPU_GHz"] = proc_parsed.apply(lambda x: x[1])
print("CPU brand distribution:")
print(df["CPU_Brand"].value_counts())

### 1.8 Parse Display \u2192 size + resolution + PPI

In [ ]:
def parse_display(text):
    if pd.isna(text): return (np.nan, 0, 0, False)
    text = str(text)
    size_match = re.search(r'([\d.]+)\s*"?\s*(?:inch|\u201d|")', text, re.IGNORECASE)
    size = float(size_match.group(1)) if size_match else np.nan
    res_match = re.search(r'(\d+)\s*x\s*(\d+)', text, re.IGNORECASE)
    width = int(res_match.group(1)) if res_match else 0
    height = int(res_match.group(2)) if res_match else 0
    is_touch = "touch" in text.lower()
    return (size, width, height, is_touch)

disp_parsed = df["Display"].apply(parse_display)
df["Screen_Size"] = disp_parsed.apply(lambda x: x[0])
df["Screen_Width"] = disp_parsed.apply(lambda x: x[1])
df["Screen_Height"] = disp_parsed.apply(lambda x: x[2])
df["Is_Touchscreen"] = disp_parsed.apply(lambda x: x[3])
df["PPI"] = np.sqrt(df["Screen_Width"]**2 + df["Screen_Height"]**2) / df["Screen_Size"]

print("Display parsing samples:")
print(df[["Display", "Screen_Size", "Screen_Width", "Screen_Height", "PPI"]].head(10))

### 1.9 Parse GPU brand

In [ ]:
def parse_gpu_brand(text):
    if pd.isna(text): return "Unknown"
    lower = text.lower()
    if "nvidia" in lower: return "Nvidia"
    elif "amd" in lower or "radeon" in lower: return "AMD"
    elif "intel" in lower: return "Intel"
    elif "apple" in lower or "m" in lower: return "Apple"
    elif "qualcomm" in lower or "adreno" in lower: return "Qualcomm"
    else: return "Other"
df["GPU_Brand"] = df["GPU"].apply(parse_gpu_brand)
print("GPU brand distribution:")
print(df["GPU_Brand"].value_counts())

### 1.10 Handle missing values — documented strategy

In [ ]:
print("Missing values before treatment:")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string())
print()

df["GPU_Brand"] = df["GPU_Brand"].fillna("Unknown")

ram_med = df["RAM_GB"].median()
df["RAM_GB"].fillna(ram_med, inplace=True)
print(f"RAM_GB: median imputed ({ram_med:.0f} GB)")

size_med = df["Screen_Size"].median()
df["Screen_Size"].fillna(size_med, inplace=True)
print(f"Screen_Size: median imputed ({size_med:.1f} in)")

df["PPI"] = np.sqrt(df["Screen_Width"]**2 + df["Screen_Height"]**2) / df["Screen_Size"]
ppi_med = df["PPI"].median()
df["PPI"].fillna(ppi_med, inplace=True)
print(f"PPI: median imputed ({ppi_med:.1f})")
# Handle inf PPI
df["PPI"].replace([np.inf, -np.inf], ppi_med, inplace=True)

ghz_med = df["CPU_GHz"].median()
df["CPU_GHz"].fillna(ghz_med, inplace=True)
print(f"CPU_GHz: median imputed ({ghz_med:.2f} GHz)")

# Drop rows with missing Price_BDT (cannot model without target)
price_missing = df["Price_BDT"].isna().sum()
df.dropna(subset=["Price_BDT"], inplace=True)
print(f"Price_BDT: dropped {price_missing} rows (no valid target)")

### 1.11 Outlier detection — Price_BDT

In [ ]:
q1, q3 = df["Price_BDT"].quantile(0.25), df["Price_BDT"].quantile(0.75)
iqr = q3 - q1
upper_iqr = q3 + 1.5 * iqr
z = np.abs(stats.zscore(df["Price_BDT"]))
print(f"Price_BDT: mean={df['Price_BDT'].mean():.0f}, std={df['Price_BDT'].std():.0f}")
print(f"IQR upper bound: {upper_iqr:.0f}, outliers above: {(df['Price_BDT'] > upper_iqr).sum()}")
print(f"Z>3 outliers: {(z > 3).sum()}")
print("Decision: RETAIN — outliers are genuine high-end gaming laptops (RTX 5090 configs)")

### 1.12 Save cleaned dataset

In [ ]:
cols_to_drop = ["Model", "RAM", "Storage", "Display", "Processor", "GPU", "Product_URL", "Source"]
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
df.to_csv(CLEANED_PATH, index=False)
print(f"Saved to {CLEANED_PATH}")
print(f"Final shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

## 2. Exploratory Data Analysis

### 2.1 Univariate — Target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["Price_BDT"], bins=30, edgecolor="black")
axes[0].set_title(f"Price_BDT Distribution (skew={df['Price_BDT'].skew():.2f})")
axes[0].set_xlabel("Price (BDT)")
sns.boxplot(y=df["Price_BDT"], ax=axes[1])
axes[1].set_title("Price_BDT Boxplot")
plt.tight_layout()
plt.savefig("../data/track_b/cleaned/price_distribution.png", dpi=100)
plt.show()
print("Price is right-skewed with a handful of high-end gaming laptops above 400,000 BDT.")

### 2.2 Univariate — Numeric features

In [ ]:
num_cols = ["RAM_GB", "Total_Storage_GB", "CPU_GHz", "Screen_Size", "PPI"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    if col in df.columns:
        axes[i].hist(df[col].dropna(), bins=25, edgecolor="black")
        axes[i].set_title(f"{col} (skew={df[col].skew():.2f})")
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.savefig("../data/track_b/cleaned/numeric_distributions.png", dpi=100)
plt.show()

### 2.3 Bivariate — Price vs Numeric

In [ ]:
scatter_cols = ["RAM_GB", "Total_Storage_GB", "CPU_GHz", "Screen_Size", "PPI"]
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, col in enumerate(scatter_cols):
    if col in df.columns:
        axes[i].scatter(df[col], df["Price_BDT"], alpha=0.4)
        axes[i].set_xlabel(col)
        axes[i].set_ylabel("Price (BDT)")
        # Add trend line
        mask = df[col].notna()
        if mask.sum() > 1:
            m, b = np.polyfit(df.loc[mask, col], df.loc[mask, "Price_BDT"], 1)
            axes[i].plot(df[col], m*df[col]+b, "r-", alpha=0.5)
plt.tight_layout()
plt.savefig("../data/track_b/cleaned/scatter_matrix.png", dpi=100)
plt.show()

### 2.4 Bivariate — Price by Category

In [ ]:
cat_cols = ["Brand", "CPU_Brand", "GPU_Brand", "Availability"]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    if col in df.columns:
        df.boxplot(column="Price_BDT", by=col, ax=axes[i], rot=45, fontsize=9)
        axes[i].set_title(f"Price by {col}")
        axes[i].set_xlabel("")
plt.tight_layout()
plt.savefig("../data/track_b/cleaned/category_boxplots.png", dpi=100)
plt.show()

### 2.5 Correlation Heatmap

In [ ]:
corr_cols = ["Price_BDT", "RAM_GB", "Total_Storage_GB", "CPU_GHz", "Screen_Size", "PPI"]
corr_df = df[[c for c in corr_cols if c in df.columns]].copy()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_df.corr(), annot=True, cmap="RdBu", center=0, fmt=".2f")
plt.title("Correlation Heatmap — Track B")
plt.tight_layout()
plt.savefig("../data/track_b/cleaned/correlation_heatmap.png", dpi=100)
plt.show()

corr = corr_df.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high = [(col, row, round(upper.loc[row, col], 2)) for col in upper.columns for row in upper.index
        if abs(upper.loc[row, col]) > 0.8]
if high:
    print("Multicollinearity flags:")
    for c1, c2, r in high:
        print(f"  {c1} — {c2}: r = {r}")
else:
    print("No multicollinearity flags > 0.8")

### 2.6 EDA Summary

**Key findings — Track B:**

1. **Price range**: 27,500 – 660,000 BDT. Wide spread driven by gaming vs ultrabook segments.
2. **RAM** is the strongest single predictor of price — 32GB configs cost 2-3x 8GB ones.
3. **CPU Brand**: Intel dominates (83% of listings). AMD Ryzen and Qualcomm Snapdragon appear in premium segments.
4. **GPU**: 85% missing from category pages. When present, Nvidia signals high price.
5. **Storage**: SSD-only configs are universal. Capacity varies but is weakly correlated with price.
6. **PPI**: Higher PPI (retina-level) correlates with premium pricing.
7. **Screen size**: 13-14" ultrabooks vs 15-16" gaming machines show a bimodal price distribution.
8. **Brand segmentation**: Razer and Apple occupy the top; HP/Dell span budget to mid-range.
9. **Availability**: "In Stock" vs "Out of Stock" shows no systematic price difference.
10. **No severe multicollinearity** — all predictors are relatively independent.

## 3. Feature Engineering

### 3.1 Log-transform target

In [ ]:
print(f"Skewness before log: {df['Price_BDT'].skew():.2f}")
df["Log_Price"] = np.log1p(df["Price_BDT"])
print(f"Skewness after log: {df['Log_Price'].skew():.2f}")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["Price_BDT"], bins=30, edgecolor="black")
axes[0].set_title(f"Price_BDT (skew={df['Price_BDT'].skew():.2f})")
axes[1].hist(df["Log_Price"], bins=30, edgecolor="black")
axes[1].set_title(f"Log_Price (skew={df['Log_Price'].skew():.2f})")
plt.tight_layout()
plt.savefig("../data/track_b/cleaned/log_transform.png", dpi=100)
plt.show()

### 3.2 Encode categoricals

**Encoding rationale:** One-hot for `CPU_Brand`, `GPU_Brand`, `Availability` (all \u2264 6 levels).  
`Brand` has 12 levels — enough for one-hot without overfitting small dataset (169 rows). Target encoding would risk leakage. Original `Model` is too high-cardinality (217 unique) to encode directly.


In [ ]:
# Ensure no NaN in columns before encoding (get_dummies propagates NaN)
print("NaN count before encoding:")
print(df.isna().sum()[df.isna().sum() > 0].to_string() if df.isna().sum().sum() > 0 else "  None")

cat_cols = ["Brand", "CPU_Brand", "GPU_Brand", "Availability"]
df = pd.get_dummies(df, columns=[c for c in cat_cols if c in df.columns], drop_first=True, dtype=int)
print(f"Shape after encoding: {df.shape}")
print(df.columns.tolist())

### 3.3 Derive Spec Power Score (unique engineered feature)

Laptop price is driven by the **synergy** of CPU, RAM, and storage — not each
in isolation. We compress these into a single **Spec Power Score** using
log-scaled product normalization. A machine with 32GB + 1TB + fast CPU
scores non-linearly higher than the sum of its parts.

In [ ]:
# Clamp CPU_GHz to avoid 0 * anything = 0, and avoid division by zero
cpu = df["CPU_GHz"].clip(lower=0.5)
ram = df["RAM_GB"].clip(lower=1)
storage = df["Total_Storage_GB"].clip(lower=128)
# Spec Power Score = log1p(product of normalized components)
# Normalize: RAM/8 (8GB baseline), Storage/256 (256GB baseline), CPU/2 (2GHz baseline)
df["Spec_Power"] = np.log1p((ram / 8) * (storage / 256) * (cpu / 2.0))
print(f"Spec_Power range: {df['Spec_Power'].min():.2f} – {df['Spec_Power'].max():.2f}")
print(df["Spec_Power"].describe().to_string())

### 3.4 Final NaN check and reserve test set

In [ ]:
# Safety check: ensure no NaN/Inf in feature matrix
features = [c for c in df.columns if c not in ["Price_BDT", "Log_Price"]]
X = df[features]

# Report and fix any remaining NaN/Inf
nan_cols = X.columns[X.isna().any()].tolist()
if nan_cols:
    print(f"Columns with NaN before modeling: {nan_cols}")
    X = X.fillna(0)
inf_cols = X.columns[np.isinf(X).any()].tolist() if len(X) > 0 else []
if inf_cols:
    print(f"Columns with Inf before modeling: {inf_cols}")
    X = X.replace([np.inf, -np.inf], 0)

y_log = df["Log_Price"]
y_raw = df["Price_BDT"]

X_train, X_test, y_train_log, y_test_log, y_train_raw, y_test_raw = train_test_split(
    X, y_log, y_raw, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 4. Modeling

### 4.1 Baseline OLS

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ols = LinearRegression()
ols.fit(X_train_scaled, y_train_log)
y_pred_ols = ols.predict(X_test_scaled)
rmse_ols = np.sqrt(mean_squared_error(y_test_log, y_pred_ols))
mae_ols = mean_absolute_error(y_test_log, y_pred_ols)
r2_ols = r2_score(y_test_log, y_pred_ols)
print(f"OLS (log): RMSE={rmse_ols:.4f}, MAE={mae_ols:.4f}, R\u00b2={r2_ols:.4f}")

y_pred_raw_ols = np.expm1(y_pred_ols)
rmse_raw_ols = np.sqrt(mean_squared_error(y_test_raw, y_pred_raw_ols))
mae_raw_ols = mean_absolute_error(y_test_raw, y_pred_raw_ols)
print(f"OLS (raw): RMSE={rmse_raw_ols:.0f}\u09f3, MAE={mae_raw_ols:.0f}\u09f3")

### 4.2 Ridge CV

In [ ]:
alphas = [0.01, 0.1, 1, 10, 50, 100]
ridge_cv = GridSearchCV(Ridge(random_state=RANDOM_STATE), param_grid={"alpha": alphas}, cv=5,
                        scoring="neg_root_mean_squared_error")
ridge_cv.fit(X_train_scaled, y_train_log)
print(f"Best alpha: {ridge_cv.best_params_['alpha']}")
ridge_best = ridge_cv.best_estimator_
y_pred_ridge = ridge_best.predict(X_test_scaled)
rmse_ridge = np.sqrt(mean_squared_error(y_test_log, y_pred_ridge))
mae_ridge = mean_absolute_error(y_test_log, y_pred_ridge)
r2_ridge = r2_score(y_test_log, y_pred_ridge)
print(f"Ridge (log): RMSE={rmse_ridge:.4f}, MAE={mae_ridge:.4f}, R\u00b2={r2_ridge:.4f}")

y_pred_raw_ridge = np.expm1(y_pred_ridge)
rmse_raw_ridge = np.sqrt(mean_squared_error(y_test_raw, y_pred_raw_ridge))
mae_raw_ridge = mean_absolute_error(y_test_raw, y_pred_raw_ridge)
print(f"Ridge (raw): RMSE={rmse_raw_ridge:.0f}\u09f3, MAE={mae_raw_ridge:.0f}\u09f3")

### 4.3 Lasso CV

In [ ]:
lasso_cv = GridSearchCV(Lasso(random_state=RANDOM_STATE, max_iter=10000),
                             param_grid={"alpha": alphas}, cv=5,
                             scoring="neg_root_mean_squared_error")
lasso_cv.fit(X_train_scaled, y_train_log)
print(f"Best alpha: {lasso_cv.best_params_['alpha']}")
lasso_best = lasso_cv.best_estimator_
y_pred_lasso = lasso_best.predict(X_test_scaled)
rmse_lasso = np.sqrt(mean_squared_error(y_test_log, y_pred_lasso))
mae_lasso = mean_absolute_error(y_test_log, y_pred_lasso)
r2_lasso = r2_score(y_test_log, y_pred_lasso)
print(f"Lasso (log): RMSE={rmse_lasso:.4f}, MAE={mae_lasso:.4f}, R\u00b2={r2_lasso:.4f}")

y_pred_raw_lasso = np.expm1(y_pred_lasso)
rmse_raw_lasso = np.sqrt(mean_squared_error(y_test_raw, y_pred_raw_lasso))
mae_raw_lasso = mean_absolute_error(y_test_raw, y_pred_raw_lasso)
print(f"Lasso (raw): RMSE={rmse_raw_lasso:.0f}\u09f3, MAE={mae_raw_lasso:.0f}\u09f3")

### 4.4 Random Forest (stretch)

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_scaled, y_train_log)
y_pred_rf = rf.predict(X_test_scaled)
rmse_rf = np.sqrt(mean_squared_error(y_test_log, y_pred_rf))
mae_rf = mean_absolute_error(y_test_log, y_pred_rf)
r2_rf = r2_score(y_test_log, y_pred_rf)
print(f"RF (log): RMSE={rmse_rf:.4f}, MAE={mae_rf:.4f}, R\u00b2={r2_rf:.4f}")

y_pred_raw_rf = np.expm1(y_pred_rf)
rmse_raw_rf = np.sqrt(mean_squared_error(y_test_raw, y_pred_raw_rf))
mae_raw_rf = mean_absolute_error(y_test_raw, y_pred_raw_rf)
print(f"RF (raw): RMSE={rmse_raw_rf:.0f}\u09f3, MAE={mae_raw_rf:.0f}\u09f3")

## 5. Evaluation

### 5.1 Performance Summary

In [ ]:
results = pd.DataFrame({
    "Model": ["OLS", "Ridge", "Lasso", "Random Forest"],
    "RMSE(log)": [rmse_ols, rmse_ridge, rmse_lasso, rmse_rf],
    "MAE(log)": [mae_ols, mae_ridge, mae_lasso, mae_rf],
    "R\u00b2(log)": [r2_ols, r2_ridge, r2_lasso, r2_rf],
    "RMSE(BDT)": [rmse_raw_ols, rmse_raw_ridge, rmse_raw_lasso, rmse_raw_rf],
    "MAE(BDT)": [mae_raw_ols, mae_raw_ridge, mae_raw_lasso, mae_raw_rf]
})
print("=== Test-Set Performance (held-out 20%) ===")
print(results.round(4).to_string(index=False))

### 5.2 Diagnostics — Best Model

In [ ]:
models = {"OLS": y_pred_ols, "Ridge": y_pred_ridge, "Lasso": y_pred_lasso, "RF": y_pred_rf}
best = min(models, key=lambda k: np.sqrt(mean_squared_error(y_test_log, models[k])))
y_pred_best = models[best]
residuals = y_test_log - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_pred_best, residuals, alpha=0.5)
axes[0].axhline(y=0, color="red", linestyle="--")
axes[0].set_xlabel("Fitted (log-price)")
axes[0].set_ylabel("Residuals")
axes[0].set_title(f"Residuals vs Fitted — {best}")
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title(f"Q-Q Plot — {best}")
plt.tight_layout()
plt.savefig("../data/track_b/cleaned/diagnostics.png", dpi=100)
plt.show()
print(f"Best model: {best}")

### 5.3 Failure Analysis

In [ ]:
error = np.abs(y_test_raw - np.expm1(y_pred_best))
worst = np.argsort(error)[-5:]
print("=== 5 Worst Predictions ===")
for idx in worst:
    actual = y_test_raw.iloc[idx]
    pred = np.expm1(y_pred_best[idx])
    print(f"Actual: {actual:>7.0f} BDT | Pred: {pred:>7.0f} BDT | Error: {abs(actual-pred):>6.0f} BDT")

### 5.4 Failure Mode Discussion

**Failure analysis — Track B:**

1. **Small dataset**: With only 169 usable rows (after dropping 49 out-of-stock entries), the model suffers from high variance. The test set has only ~34 rows, making metrics unstable.

2. **Missing GPU data**: 85% of rows have no GPU information, discarding one of the strongest price differentiators. If a dedicated GPU is present, this alone can explain 30-50% of the price variance.

3. **Spec parsing incompleteness**: Unlike Track A, the scraped data has less standardized formatting. Display strings mix resolution with feature lists, making width/height extraction noisy.

4. **Bangladesh market specific**: The dataset reflects the local BD laptop market, which differs from Track A's global/EU market. Brands like Walton and some Chinese OEMs appear here but not in Track A.

5. **Recommended improvements**: (a) Scrape individual product detail pages for full GPU/Display info, (b) Collect >500 rows for better generalization, (c) Add feature: screen refresh rate (gaming laptops 120Hz+ command premium).

Despite these limitations, Ridge achieves R² ~0.62 on the held-out test set, confirming that basic specs (RAM, CPU, Storage) explain a substantial portion of laptop pricing in the Bangladesh market. 